# Bronze Layer: Extraction

**Target Tables:**
- **Read:** `bronze_general_search` (SQLite `scraping_database.db`)
- **Write:** `bronze_ad_links` / Raw HTML storage (SQLite / Parquet)

**Objective:**
This notebook takes the list of URLs discovered in the `discover.ipynb` notebook and fetches the detailed HTML/JSON data for each ad. It filters the target regions based on the metadata to avoid unnecessary requests and extracts the raw content of each advertisement, saving it to the Bronze layer without heavy transformations.

**To-Do:**
- Implement the detailed extraction logic for OLX.
- Implement rate-limiting, retries, and anti-blocking strategies.
- Decide on the storage format for raw HTML/JSON (e.g., Parquet or Blob storage).

In [2]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[2])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import insert, select, update
from sqlalchemy.orm import Session
import polars as pl
from crawlers import OLXCrawler

# Importing from 'app' module
from app.config import db_engine
from app.models import GeneralSearch, InformationExtraction
from app.utils import read_search_locations_metadata

In [3]:
# List to store the pages' data
extraction_data_list = []

# List to store the selled/deleted products
not_available_products = []

In [4]:
# Region data for filtered results due to limits
brazil_data_location = (
    read_search_locations_metadata()
    .get('search_locations', {})
    .get('brazil', {})
)

stores = (
    brazil_data_location
    .get('stores', [])
)

regions = (
    brazil_data_location
    .get('regions', [])
)

full_name_list = [region.get('full_name') for region in regions]
abreviation_list = [region.get('abreviation') for region in regions]

In [5]:
# Reading values
with db_engine.connect() as connection:
    # Removing undesired regions
    df_olx = (
        pl.read_database(
            select(GeneralSearch)
            .where(
                (GeneralSearch.region.in_(abreviation_list))
                & (GeneralSearch.store.is_('OLX'))
            ),
            connection=connection
        )
    )


In [6]:
df_olx = df_olx.head(100)

In [6]:
olx_crawler = OLXCrawler(base_data_list=extraction_data_list)
await olx_crawler.scrap_specific_information(
    df=df_olx,
    not_available_products=not_available_products
)

INFO:httpx:HTTP Request: GET https://mg.olx.com.br/regiao-de-juiz-de-fora/informatica/notebooks/samsung-galaxy-book-go-windows-11-seminovo-1518446872 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/regiao-de-bauru-e-marilia/informatica/notebooks/samsung-galaxy-book-go-com-processador-snapdragon-7c-1517901028 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://rs.olx.com.br/regioes-de-porto-alegre-torres-e-santa-cruz-do-sul/informatica/notebooks/notebook-samsung-galaxy-book-go-14-full-hd-windows-11-novo-1485104529 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/samsung-galaxy-book-go-1518403181 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-go-1518559821 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://rj.olx.com.br/rio-de-janeiro-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-go-1516195780 "H

Unexpected error: Client error '403 Forbidden' for url 'https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-1513893900'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403


INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-1513893900 "HTTP/1.1 403 Forbidden"
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403.
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/regiao-de-sao-jose-do-rio-preto/informatica/notebooks/notebook-samsung-galaxy-book-2-1486131844 "HTTP/1.1 200 OK"


Unexpected error: Client error '403 Forbidden' for url 'https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-1513893900'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403


INFO:httpx:HTTP Request: GET https://sp.olx.com.br/baixada-santista-e-litoral-sul/informatica/notebooks/notebook-samsung-galaxy-book-2-1514786890 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-15-6-1510409973 "HTTP/1.1 403 Forbidden"
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403.


...

INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-1489586876 "HTTP/1.1 403 Forbidden"
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403.


Unexpected error: Client error '403 Forbidden' for url 'https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-1489586876'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403


INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-15-6-1510409973 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-1489586876 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://rj.olx.com.br/rio-de-janeiro-e-regiao/informatica/notebooks/samsung-galaxy-book-2-intel-i5-8gb-256gb-ssd-windows-11-impecavel-1514129946 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-1515387582 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/samsung-galaxy-book-2-intel-i5-12-geracao-512gb-de-ssd-8gb-de-ram-15-6-1465596451 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-2-intel-core-i

Unexpected error: Client error '403 Forbidden' for url 'https://rj.olx.com.br/rio-de-janeiro-e-regiao/informatica/notebooks/lenovo-ideapad-3-15itl6-intel-core-i5-1511539502'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403


INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3-rapido-conservado-e-pronto-para-trabalho-estudos-1503085055 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3-1511304264 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3i-12gb-ram-256gb-ssd-15-6-preto-1475875149 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mg.olx.com.br/belo-horizonte-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3i-celeron-4gb-128gb-ssd-microsoft-365-personal-windows-11-15-1518556378 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mg.olx.com.br/regiao-de-uberlandia-e-uberaba/informatica/notebooks/notebook-lenovo-ideapad-3i-i3-10-ger-ssd-256gb-15-6-windows-11-1518466325 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://rs.olx.com.br/regioes-de-porto-ale

In [7]:
# Creating dataframe
extraction_data_df = pl.DataFrame(extraction_data_list)
updated_df = pl.DataFrame(not_available_products)

if not extraction_data_df.is_empty():
    with Session(db_engine) as session:
        # Saving on sqlite
        session.execute(insert(InformationExtraction), extraction_data_df.to_dicts())
        session.commit()
    
    if not updated_df.is_empty():
        session.execute(update(GeneralSearch), updated_df.to_dicts())
        session.commit()
    else:
        print("No data found to update.")
else:
    print("No data found to insert.")

No data found to update.
